# Capstone 2 — Domain-Specialist Mini-LLM

You will fine-tune a 7B base model with **QLoRA** on a domain dataset of your choice (legal clauses, medical Q&A, SQL generation, your own customer-support transcripts), quantize the result to INT4 GGUF, and serve it locally via Ollama. Then you will *prove* it improved with an eval suite.

This capstone makes you actually *do* the workflow that fine-tuning blog posts hand-wave through.

## Phase 0 — Concept Recall

Answer in markdown:
1. Why does QLoRA fit a 7B model on a 16 GB GPU?
2. What's the failure mode if your `r` (LoRA rank) is too low? Too high?
3. Pick a quantization scheme for serving and justify it for *your* deployment target.

*(Each = 5 pts. Vague answers lose marks; cite a number or a tradeoff.)*

In [ ]:
# Phase 1 — Environment
# Recommended: Colab T4 (free) or a local 16GB+ GPU.
# pip install transformers peft bitsandbytes accelerate datasets trl
#
# If no GPU is available, switch BASE_MODEL to a 1B–3B model
# (e.g. Qwen2.5-1.5B) so this still runs end-to-end.

import torch, os
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE_MODEL = "mistralai/Mistral-7B-v0.3"   # or "Qwen/Qwen2.5-1.5B"
DOMAIN     = "sql_generation"               # change to your chosen domain

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                          bnb_4bit_compute_dtype=torch.bfloat16)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map="auto")
model = prepare_model_for_kbit_training(model)


In [ ]:
# Phase 2 — Dataset (Guided Build)
# TODO 1: Load YOUR domain dataset. Examples:
#   ds = load_dataset("b-mc2/sql-create-context", split="train")
#   ds = load_dataset("openai_humaneval", split="test")
# Map each row to a single 'text' field with a consistent template.

PROMPT_TEMPLATE = """### Instruction:
{instruction}
### Input:
{input}
### Response:
{output}"""

def format_example(row):
    # TODO 2: adapt to your dataset's column names
    return {"text": PROMPT_TEMPLATE.format(
        instruction=row.get("question", ""),
        input=row.get("context", ""),
        output=row.get("answer", ""),
    )}

# ds = ds.map(format_example).select(range(2000))   # cap to fit your time budget


In [ ]:
# Phase 3 — LoRA config + train
lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()   # should show <1% trainable

from trl import SFTConfig, SFTTrainer
cfg = SFTConfig(
    output_dir="./out",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    save_steps=200,
    max_seq_length=1024,
    dataset_text_field="text",
)
# trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds, tokenizer=tokenizer)
# trainer.train()
# trainer.save_model("./out/final")


In [ ]:
# Phase 4 — Eval (the part most tutorials skip)
# Build a held-out test set BEFORE training; never let it leak into training.

TEST_SET = [
    # ("instruction", "expected_substring_or_exact_match"),
    # TODO 3: 50+ examples, hand-curated.
]

def generate(model, tokenizer, instruction: str, max_new_tokens=128):
    # TODO 4: call model.generate with the same prompt template used in training
    raise NotImplementedError

def score(model, tokenizer):
    hits = 0
    for instr, expected in TEST_SET:
        out = generate(model, tokenizer, instr)
        if expected.lower() in out.lower():
            hits += 1
    return hits / len(TEST_SET)

# baseline_acc  = score(base_model, tokenizer)   # before FT
# finetuned_acc = score(model, tokenizer)        # after FT
# print("lift:", finetuned_acc - baseline_acc)


In [ ]:
# Phase 5 — Catastrophic-forgetting check (independent extension)
# A great FT improves the target task without crippling general ability.
# Run a small generic benchmark before and after; lift on target should
# not be paid for with a >5% drop on generic tasks.

GENERIC_PROMPTS = [
    "What is the capital of Brazil?",
    "Write a haiku about the sea.",
    "Explain photosynthesis to a 10-year-old.",
]
# TODO 5: judge each output before/after FT with an LLM-as-judge.


In [ ]:
# Phase 6 — Quantize + serve
# After training, merge adapters, export GGUF, and serve via Ollama.
#
# # merge
# model = model.merge_and_unload()
# model.save_pretrained("./merged")
#
# # convert (CLI, run outside notebook):
# # python llama.cpp/convert.py ./merged --outfile model.gguf
# # ./llama.cpp/quantize model.gguf model.Q4_K_M.gguf Q4_K_M
#
# # serve:
# # ollama create my-specialist -f Modelfile
# # ollama run my-specialist


## Phase 7 — Submission checklist (rubric / 100)

- [ ] **30 pts** — `finetuned_acc - baseline_acc ≥ 0.10` on your held-out test set.
- [ ] **15 pts** — Reproducible training: `python train.py` from clean clone produces the same artefacts (seed pinned, requirements frozen).
- [ ] **15 pts** — Inference latency on a Q4_K_M GGUF, measured (tokens/sec, p50 + p95).
- [ ] **15 pts** — Catastrophic-forgetting evaluation passes (≤ 5% drop on generic prompts).
- [ ] **15 pts** — Run logged to W&B / MLflow / a JSON file; loss curve attached.
- [ ] **10 pts** — Model card: dataset provenance, license, known limitations, intended use.

### What you will have *learned by doing*
- That LoRA actually trains in minutes on consumer hardware.
- That dataset quality dominates everything else.
- That "fine-tuned" without an eval is just vibes.